# M12 lab solution: Chunking Strategies

**Brief.** Chunking bake-off: index the same corpus with recursive, semantic and proposition chunking at three sizes each, and compare hit rate and answer faithfulness against the M10 baseline.

Run with the default environment: `uv run jupyter lab`.

Open this notebook from `lab/solution/`, so `../data/` resolves to the lab's own data folder.

## Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # API keys from the .env file at the repo root
sys.path.insert(0, str(Path("../../../../src").resolve()))

from lab_kit import baseline, hit_rate, is_hit, load_handbook, load_jsonl, mrr, recall_at_k  # noqa: E402

DATA = Path("../data/larkspur")
handbook = load_handbook(DATA / "handbook")
golden = load_jsonl(DATA / "golden.jsonl")
by_id = {g["id"]: g for g in golden}
scored = [g for g in golden if g["evidence_spans"]]
print(len(handbook), "handbook files,", len(golden), "golden questions,", len(scored), "with evidence")

In [ ]:
from helpers import get_embeddings, get_experientiallabs_llm

llm = get_experientiallabs_llm()
embeddings = get_embeddings()
LLM_NAME = llm.model_name
EMBEDDING_NAME = getattr(embeddings, "model", None) or getattr(embeddings, "endpoint", None)
print("LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)

In [ ]:
# The inline "Part V system", reused by the later labs: the handbook split as lab_kit.baseline splits it
# (recursive, 500 characters, 50 overlap), a FAISS vector retriever over those chunks, and an answer
# prompt that cites numbered passages and is allowed to say "I don't know".
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [
    Document(page_content=c, metadata={"source": p["source"], "section": p["title"]})
    for p in handbook
    for c in splitter.split_text(p["text"])
]

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions from Larkspur Cloud customers using only the numbered handbook passages. "
               "Cite the passages you use as [1], [2]. If the passages do not contain the answer, "
               "reply exactly: I don't know."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{i}] ({d.metadata.get('source', '?')}) {d.page_content}" for i, d in enumerate(docs, 1))


def answer(question, docs, config=None):
    return (ANSWER_PROMPT | llm).invoke({"context": format_docs(docs), "question": question}, config=config).content


print(len(chunks), "baseline chunks")

In [ ]:
def texts(docs):
    return [d.page_content for d in docs]


def score(results, questions=None):
    """results: question id -> retrieved chunk texts in rank order; scored with lab_kit on the questions with evidence."""
    qs = questions or golden
    return {"hit_rate": round(hit_rate(results, qs), 3), "recall_at_k": round(recall_at_k(results, qs), 3),
            "mrr": round(mrr(results, qs), 3)}

In [ ]:
from pydantic import BaseModel, Field


class Grade(BaseModel):
    correct: bool = Field(description="True when the answer states every fact the reference answer gives for this question.")
    faithful: bool = Field(description="True when every claim in the answer is supported by the passages.")


JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You grade one answer from a customer-support RAG system. Judge it against the reference answer "
               "and the passages the system was given. Ignore citation markers such as [1]."),
    ("human", "Question: {question}\n\nReference answer: {reference}\n\nPassages:\n{context}\n\nAnswer to grade: {answer}"),
])
judge = JUDGE_PROMPT | llm.with_structured_output(Grade)


def grade(g, docs, reply):
    return judge.invoke({"question": g["question"], "reference": g["reference_answer"],
                         "context": format_docs(docs), "answer": reply})


def is_refusal(reply):
    return reply.strip().lower().startswith("i don't know")

## 1. Nine chunkings

- **Recursive** at 250, 500 and 1,000 characters (overlap 10%).
- **Semantic** (`SemanticChunker`, percentile breakpoints) at the 80th, 90th and 95th percentile: a lower percentile cuts more often, so it gives smaller chunks.
- **Propositions**: an LLM rewrites each 500-character recursive chunk as atomic facts (one call per chunk, paid once), then 1, 3 or 6 consecutive propositions from the same base chunk form one index entry.

Propositions are rewritten text, so they rarely contain an evidence span word for word. A proposition entry is scored by the text of the base chunk it came from (stored as `parent`): a hit means retrieval found the right passage, which is what the span check asks of every other chunker.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker

SOURCES = [Document(page_content=p["text"], metadata={"source": p["source"], "section": p["title"]}) for p in handbook]


def recursive(size):
    s = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 10)
    return s.split_documents(SOURCES)


def semantic(percentile):
    s = SemanticChunker(embeddings, breakpoint_threshold_type="percentile", breakpoint_threshold_amount=percentile)
    return s.split_documents(SOURCES)


chunkings = {f"recursive {n}": recursive(n) for n in (250, 500, 1000)}
chunkings.update({f"semantic p{p}": semantic(p) for p in (80, 90, 95)})
for name, docs in chunkings.items():
    print(f"{name:15} {len(docs):3} chunks, mean {sum(len(d.page_content) for d in docs) / len(docs):6.0f} chars")

In [ ]:
from pydantic import BaseModel, Field


class Propositions(BaseModel):
    propositions: list[str] = Field(description="Atomic, self-contained facts. Each names its subject and can be read alone.")


PROP_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Break the passage into propositions: short, self-contained statements of one fact each. "
               "Replace pronouns with the names they refer to. Use only facts stated in the passage."),
    ("human", "Section: {section}\n\nPassage:\n{passage}"),
])
proposer = PROP_PROMPT | llm.with_structured_output(Propositions)

base = recursive(500)
props_by_base = []
for i, d in enumerate(base):
    out = proposer.invoke({"section": d.metadata["section"], "passage": d.page_content})
    props_by_base.append(out.propositions)
print(sum(map(len, props_by_base)), "propositions from", len(base), "base chunks")


def proposition_chunks(group):
    out = []
    for d, props in zip(base, props_by_base):
        for j in range(0, len(props), group):
            out.append(Document(page_content=" ".join(props[j:j + group]),
                                metadata={**d.metadata, "parent": d.page_content}))
    return out


chunkings.update({f"propositions x{n}": proposition_chunks(n) for n in (1, 3, 6)})
print({name: len(docs) for name, docs in chunkings.items()})

## 2. Index, retrieve and score

Each chunking gets its own FAISS index and the same top-4 retrieval. Hit rate, recall and MRR come from `lab_kit` on the 25 questions with evidence. Answer faithfulness: each question is answered from its top 4 entries (the proposition text itself, which is what a proposition index would send to the model) and judged against those passages.

In [ ]:
K = 4


def scored_text(doc):
    return doc.metadata.get("parent", doc.page_content)


rows = []
for name, docs in chunkings.items():
    store = FAISS.from_documents(docs, embeddings)
    hits = {g["id"]: store.similarity_search(g["question"], k=K) for g in golden}
    metrics = score({qid: [scored_text(d) for d in ds] for qid, ds in hits.items()})
    faithful = []
    for g in scored:
        reply = answer(g["question"], hits[g["id"]])
        faithful.append(grade(g, hits[g["id"]], reply).faithful)
    rows.append({"chunking": name, "chunks": len(docs), **metrics, "faithfulness": round(sum(faithful) / len(faithful), 3)})
    print(rows[-1])

## Results against the baseline

In [ ]:
import pandas as pd

ref = baseline(DATA, k=K)
table = pd.DataFrame(rows + [{"chunking": "lab_kit.baseline (TF-IDF, recursive 500)", "chunks": None,
                              "hit_rate": round(ref["hit_rate"], 3), "recall_at_k": round(ref["recall_at_k"], 3),
                              "mrr": round(ref["mrr"], 3), "faithfulness": None}])
print("run:", time.strftime("%Y-%m-%d"), "| LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)
print(table.to_string(index=False))

## What this shows

- Read each family down its three sizes first: the trend with size says more than any single cell. Small chunks retrieve precisely but can cut a fact in two; large ones keep facts whole and send more tokens per hit.
- Semantic chunking cuts where neighbouring sentences drift apart, at a threshold computed from each document's own distances, so on short, single-topic handbook sections it may cut very little. Check the chunk counts before reading its scores.
- Propositions cost one LLM call per base chunk at index time and score retrieval through their parent passage. Their faithfulness column is the price of rewriting: an answer can only be as faithful as the propositions the LLM wrote.